# VOC segmentation - Colab training runs

Runs the three `--arch` variants on a free-tier GPU and produces the numbers
for the README results table and the Tier 4 ablation table.

**Before you start:** Runtime -> Change runtime type -> Hardware accelerator = **T4 GPU**.

Run the cells top to bottom. Cell 3 (dataset) is the slow one the first time
(~2 GB download); after that it is cached on your Google Drive.

## 1. Confirm you actually got a GPU

In [ ]:
!nvidia-smi
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'

## 2. Get the code

Do **not** `pip install -r requirements.txt` here. Colab already ships torch and
torchvision built against its CUDA driver; installing the CPU-era pins from
`requirements.txt` (torch 2.5.1) would either downgrade you or pull a wheel that
does not match the runtime. The training code only uses stable APIs
(`torch.amp.GradScaler`, `torch.autocast`, `torchvision.datasets.VOCSegmentation`),
so Colab's newer versions are fine.

In [ ]:
%cd /content
!rm -rf voc-semantic-segmentation
!git clone --branch fix/segmentation-overhaul https://github.com/karinamoffat/voc-semantic-segmentation.git
%cd /content/voc-semantic-segmentation
!git log --oneline -1

## 3. Pascal VOC 2012, cached on Drive

`modelSS_train.py` hardcodes `root='./data'` with `download=True`. torchvision
skips the download when `data/VOCtrainval_11-May-2012.tar` is already present
with a matching md5 - so we keep that one tar on your Drive and copy it in at
the start of every session instead of re-downloading 2 GB from
`host.robots.ox.ac.uk`, which is slow and frequently down.

Drive cost: ~2 GB of your 15 GB free quota.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

CACHE = '/content/drive/MyDrive/voc2012'
TAR = 'VOCtrainval_11-May-2012.tar'
URL = 'http://host.robots.ox.ac.uk/pascal/VOC/voc2012/' + TAR
os.makedirs(CACHE, exist_ok=True)
os.makedirs('data', exist_ok=True)

if not os.path.exists(f'{CACHE}/{TAR}'):
    print('First run: downloading ~2 GB to Drive (once).')
    !wget -c -O "$CACHE/$TAR" "$URL"
else:
    print('Using cached tar on Drive.')

shutil.copy(f'{CACHE}/{TAR}', f'data/{TAR}')
!ls -lh data/

In [ ]:
# Sanity check: md5 must match torchvision's expected hash or it re-downloads.
!md5sum data/VOCtrainval_11-May-2012.tar
print('expected: 6cd6e144f989b92b3379bac3b3de84fd')

## 4. Short shakedown run

Two epochs first. This proves the data path, the GPU path and AMP all work, and
gives you a real per-epoch wall-clock number (the script logs it) so you can
budget the full runs before committing an hour of session time to them.

In [ ]:
!python modelSS_train.py --arch baseline -e 2 -b 16 --num-workers 2 --amp --augment \n    -w /content/drive/MyDrive/voc2012/smoke.pth -p smoke.png

## 5. The three ablation runs

Same seed, same split, same epochs - that is what makes the Tier 4 table a fair
comparison. Weights and plots are written straight to Drive so an idle
disconnect does not cost you the run.

Run these one cell at a time and note the **best val mIoU** each one logs.

In [ ]:
OUT = '/content/drive/MyDrive/voc2012/runs'
import os; os.makedirs(OUT, exist_ok=True)
COMMON = '-e 30 -b 16 --num-workers 2 --amp --augment --seed 0'
print(COMMON)

In [ ]:
!python modelSS_train.py --arch baseline $COMMON -w $OUT/baseline.pth -p baseline.png

In [ ]:
!python modelSS_train.py --arch unet $COMMON -w $OUT/unet.pth -p unet.png

In [ ]:
!python modelSS_train.py --arch resnet18 $COMMON -w $OUT/resnet18.pth -p resnet18.png

## 6. Qualitative grid

Use whichever arch won. `predict.py` writes `results/qualitative.png` and creates
the directory itself.

In [ ]:
BEST_ARCH = 'resnet18'  # <- set this to whichever won
!python predict.py --arch $BEST_ARCH -w $OUT/$BEST_ARCH.pth -n 6 -o results/qualitative.png

from IPython.display import Image, display
display(Image('results/qualitative.png'))

## 7. Pull the artifacts back to your machine

In [ ]:
!cp results/qualitative.png loss_*.png mIoU_*.png $OUT/
!ls -lh $OUT

# Or download directly through the browser:
from google.colab import files
files.download('results/qualitative.png')